In [4]:
import os
import numpy as np
import xarray as xr
import rasterio
from rasterio.transform import xy
from rasterio.transform import from_origin
from glob import glob

In [1]:
!pwd

/Users/workad/Workspace/Article3-AtmosphericDynamics/RecoveredNB


In [ ]:
# Step 1: Set path to TIF files
tif_folder = "../output/GPH850"
tif_files = sorted(glob(os.path.join(tif_folder, "*.tif")))

# Step 2: Extract dates from filenames
dates = [os.path.basename(f).replace(".tif", "") for f in tif_files]  # ['DJF 1991', 'MAM 1991', ..., 'SON 2024']

# Step 3: Read one file to get spatial dimensions and transform

with rasterio.open(tif_files[0]) as src:
    height, width = src.height, src.width
    transform = src.transform
    crs = src.crs

    # Generate coordinate values from row and column indices
    row_indices = np.arange(height)
    col_indices = np.arange(width)

    # Get center coordinates
    lon, _ = xy(transform, np.zeros_like(col_indices), col_indices, offset="center")
    _, lat = xy(transform, row_indices, np.zeros_like(row_indices), offset="center")

    lon = np.array(lon)
    lat = np.array(lat)

    if lat[0] > lat[-1]:
        lat = lat[::-1]
        reverse_lat = True
    else:
        reverse_lat = False

# Ensure latitude is in ascending order (as required by xarray)
if lat[0] > lat[-1]:
    lat = lat[::-1]
    reverse_lat = True
else:
    reverse_lat = False

# Step 4: Read all data into a numpy array
data = []
for file in tif_files:
    with rasterio.open(file) as src:
        array = src.read(1)  # Read first band
        if reverse_lat:
            array = np.flipud(array)
        data.append(array)

data = np.stack(data)  # Shape: (137, 401, 521)

# Step 5: Create xarray DataArray
da = xr.DataArray(
    data,
    dims=("date", "latitude", "longitude"),
    coords={
        "date": dates,
        "latitude": lat,
        "longitude": lon
    },
    name="GPH850_anomalies"
)

# Step 6: Save to NetCDF
da.to_netcdf("../output/GPH850_anomalies.nc")

In [7]:
# Load data 
ds = xr.open_dataset("../output/GPH200_anomalies.nc") 
da = ds['GPH200_anomalies'] # shape: (137, lat, lon) 
dates = da.coords['date'].values 
data = da.values 
n_time, n_lat, n_lon = data.shape 
# Spatial transform 
lat = da.latitude.values 
lon = da.longitude.values 
pixel_size_x = abs((lon[-1] - lon[0])) / (len(lon) - 1) 
pixel_size_y = abs((lat[-1] - lat[0])) / (len(lat) - 1) 
transform = from_origin( lon[0] - pixel_size_x / 2, lat[0] - pixel_size_y / 2, pixel_size_x, -pixel_size_y ) 
# Output directory 
output_dir = "../output/GPH200_percentile_significance" 
os.makedirs(output_dir, exist_ok=True)
# Loop over seasons 
for t in range(n_time): 
    anomaly = data[t, :, :] # Anomaly map for this season 
    significance = np.zeros((n_lat, n_lon), dtype=np.int8) # -1, 0, +1 
    for i in range(n_lat): 
        for j in range(n_lon): 
            series = data[:, i, j] 
            x_t = series[t] 
            if np.isnan(x_t): 
                continue 
            x_rest = np.delete(series, t) 
            x_rest = x_rest[~np.isnan(x_rest)] 
            if len(x_rest) < 10: 
                continue # too little data for percentiles 
            p5 = np.percentile(x_rest, 5) 
            p95 = np.percentile(x_rest, 95) 
            if x_t < p5: 
                significance[i, j] = -1 
            elif x_t > p95: 
                significance[i, j] = 1 # else: leave as 0 
    # Save GeoTIFF 
    filename = f"{dates[t]}.tif" 
    filepath = os.path.join(output_dir, filename) 
    with rasterio.open( filepath, 'w', driver='GTiff', height=n_lat, width=n_lon, count=2, dtype=rasterio.float32, crs="EPSG:4326", transform=transform ) as dst: 
        dst.write(anomaly.astype(np.float32), 1) # Band 1: anomaly 
        dst.write(significance.astype(np.float32), 2) # Band 2: significance mask 
    print(f"Saved: {filename}")

Saved: DJF 1991.tif
Saved: DJF 1992.tif
Saved: DJF 1993.tif
Saved: DJF 1994.tif
Saved: DJF 1995.tif
Saved: DJF 1996.tif
Saved: DJF 1997.tif
Saved: DJF 1998.tif
Saved: DJF 1999.tif
Saved: DJF 2000.tif
Saved: DJF 2001.tif
Saved: DJF 2002.tif
Saved: DJF 2003.tif
Saved: DJF 2004.tif
Saved: DJF 2005.tif
Saved: DJF 2006.tif
Saved: DJF 2007.tif
Saved: DJF 2008.tif
Saved: DJF 2009.tif
Saved: DJF 2010.tif
Saved: DJF 2011.tif
Saved: DJF 2012.tif
Saved: DJF 2013.tif
Saved: DJF 2014.tif


KeyboardInterrupt: 